This is a companion notebook for the book [Deep Learning with Python, Third Edition](https://www.manning.com/books/deep-learning-with-python-third-edition). For readability, it only contains runnable code blocks and section titles, and omits everything else in the book: text paragraphs, figures, and pseudocode.

**If you want to be able to follow what's going on, I recommend reading the notebook side by side with your copy of the book.**

The book's contents are available online at [deeplearningwithpython.io](https://deeplearningwithpython.io).

In [ ]:
!pip install torch torchvision --upgrade -q

## A deep dive on Keras

### A spectrum of workflows

### Different ways to build Keras models

#### The Sequential model

In [ ]:
import torch
from torch import nn

# PyTorch: nn.Sequential needs explicit in/out sizes (no lazy build) and we drop
# the final softmax, since classification losses here expect raw logits.
model = nn.Sequential(
    nn.LazyLinear(64),
    nn.ReLU(),
    nn.Linear(64, 10),
)

In [ ]:
# PyTorch: nn.Sequential takes modules positionally; there's no .add(), so we
# build the same stack by listing the layers. LazyLinear infers in_features
# on first forward pass (the closest analog to Keras's deferred build).
model = nn.Sequential(
    nn.LazyLinear(64),
    nn.ReLU(),
    nn.Linear(64, 10),
)

In [ ]:
# PyTorch: weights live in model.parameters(); with Lazy layers they're
# uninitialized until the first forward pass, mirroring Keras's unbuilt state.
list(model.parameters())

In [ ]:
# PyTorch: there's no build(); run a dummy batch through the model to
# materialize the Lazy layers' weights, then inspect them.
model(torch.zeros(1, 3))
[p.shape for p in model.parameters()]

In [ ]:
# PyTorch: print(model) is the closest equivalent to Keras's summary().
print(model)

In [ ]:
# PyTorch: modules can be named by passing an OrderedDict to nn.Sequential.
from collections import OrderedDict

model = nn.Sequential(OrderedDict([
    ("my_first_layer", nn.Linear(3, 64)),
    ("my_first_relu", nn.ReLU()),
    ("my_last_layer", nn.Linear(64, 10)),
]))
print(model)

In [ ]:
# PyTorch: specifying the input shape up front just means giving in_features
# explicitly (nn.Linear(3, 64)) instead of using a separate Input layer.
model = nn.Sequential(
    nn.Linear(3, 64),
    nn.ReLU(),
)

In [ ]:
print(model)

In [ ]:
# PyTorch: extend the stack by rebuilding it with the extra layer.
model = nn.Sequential(
    nn.Linear(3, 64),
    nn.ReLU(),
    nn.Linear(64, 10),
)
print(model)

#### The Functional API

##### A simple example

In [ ]:
# PyTorch: the Functional API (keras.Model(inputs, outputs)) becomes an
# nn.Module subclass whose forward() wires the layers together.
class MyFunctionalModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.dense_1 = nn.Linear(3, 64)
        self.dense_2 = nn.Linear(64, 10)

    def forward(self, inputs):
        features = torch.relu(self.dense_1(inputs))
        outputs = self.dense_2(features)
        return outputs

model = MyFunctionalModel()

In [ ]:
# PyTorch: there's no symbolic Input object; an "input" is just a real tensor
# with a given shape. We use a dummy batch of one sample to illustrate.
inputs = torch.zeros(1, 3)

In [ ]:
inputs.shape

In [ ]:
inputs.dtype

In [ ]:
# PyTorch: layers are modules you instantiate and then call on a tensor.
dense = nn.Linear(3, 64)
features = torch.relu(dense(inputs))

In [ ]:
features.shape

In [ ]:
# PyTorch: chain the final layer and wrap the whole graph in an nn.Module.
outputs = nn.Linear(64, 10)(features)

class MyFunctionalModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.dense_1 = nn.Linear(3, 64)
        self.dense_2 = nn.Linear(64, 10)

    def forward(self, inputs):
        return self.dense_2(torch.relu(self.dense_1(inputs)))

model = MyFunctionalModel()

In [ ]:
print(model)

##### Multi-input, multi-output models

In [ ]:
vocabulary_size = 10000
num_tags = 100
num_departments = 4

# PyTorch: a multi-input/multi-output functional model becomes an nn.Module that
# takes the three inputs and returns the two outputs. Concatenate -> torch.cat.
# We drop the sigmoid/softmax output activations and return raw logits, since the
# losses below (BCEWithLogitsLoss / CrossEntropyLoss) expect logits.
class TicketClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.dense_features = nn.Linear(
            vocabulary_size + vocabulary_size + num_tags, 64
        )
        self.priority = nn.Linear(64, 1)
        self.department = nn.Linear(64, num_departments)

    def forward(self, title, text_body, tags):
        features = torch.cat([title, text_body, tags], dim=1)
        features = torch.relu(self.dense_features(features))
        priority = self.priority(features)
        department = self.department(features)
        return priority, department

model = TicketClassifier()

##### Training a multi-input, multi-output model

In [ ]:
import numpy as np

num_samples = 1280

title_data = np.random.randint(0, 2, size=(num_samples, vocabulary_size))
text_body_data = np.random.randint(0, 2, size=(num_samples, vocabulary_size))
tags_data = np.random.randint(0, 2, size=(num_samples, num_tags))

priority_data = np.random.random(size=(num_samples, 1))
department_data = np.random.randint(0, num_departments, size=(num_samples, 1))

# PyTorch: compile() -> create an optimizer and the two loss functions. The
# priority head uses BCEWithLogitsLoss (binary), the department head uses
# CrossEntropyLoss (multiclass), matching mse->mae and softmax+xent originals.
optimizer = torch.optim.Adam(model.parameters())
priority_loss_fn = nn.BCEWithLogitsLoss()
department_loss_fn = nn.CrossEntropyLoss()

# PyTorch: convert NumPy arrays to tensors once.
title_t = torch.from_numpy(title_data).float()
text_body_t = torch.from_numpy(text_body_data).float()
tags_t = torch.from_numpy(tags_data).float()
priority_t = torch.from_numpy(priority_data).float()
department_t = torch.from_numpy(department_data).long().squeeze(1)

# PyTorch: fit() -> explicit training loop. The total loss is the sum of the two
# head losses (Keras sums per-output losses by default).
batch_size = 32
for epoch in range(1):
    permutation = torch.randperm(num_samples)
    for i in range(0, num_samples, batch_size):
        idx = permutation[i : i + batch_size]
        priority_pred, department_pred = model(
            title_t[idx], text_body_t[idx], tags_t[idx]
        )
        loss = priority_loss_fn(priority_pred, priority_t[idx]) + department_loss_fn(
            department_pred, department_t[idx]
        )
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

# PyTorch: evaluate() -> manual no-grad pass reporting the same metrics (MAE for
# priority, accuracy for department).
model.eval()
with torch.no_grad():
    priority_pred, department_pred = model(title_t, text_body_t, tags_t)
    mae = (torch.sigmoid(priority_pred) - priority_t).abs().mean().item()
    dept_acc = (department_pred.argmax(dim=1) == department_t).float().mean().item()
print(f"priority MAE: {mae:.4f}, department accuracy: {dept_acc:.4f}")

# PyTorch: predict() -> no-grad forward; apply sigmoid/softmax to get probabilities.
with torch.no_grad():
    priority_logits, department_logits = model(title_t, text_body_t, tags_t)
    priority_preds = torch.sigmoid(priority_logits).numpy()
    department_preds = torch.softmax(department_logits, dim=1).numpy()
model.train()

In [ ]:
# PyTorch: the dict-keyed compile/fit form (loss/metrics keyed by output name)
# is, in PyTorch, just a matter of naming the per-head losses ourselves. The
# training loop is identical to the list form above.
losses = {
    "priority": nn.BCEWithLogitsLoss(),
    "department": nn.CrossEntropyLoss(),
}
optimizer = torch.optim.Adam(model.parameters())

batch_size = 32
for epoch in range(1):
    permutation = torch.randperm(num_samples)
    for i in range(0, num_samples, batch_size):
        idx = permutation[i : i + batch_size]
        priority_pred, department_pred = model(
            title_t[idx], text_body_t[idx], tags_t[idx]
        )
        loss = losses["priority"](priority_pred, priority_t[idx]) + losses[
            "department"
        ](department_pred, department_t[idx])
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

model.eval()
with torch.no_grad():
    priority_logits, department_logits = model(title_t, text_body_t, tags_t)
    priority_preds = torch.sigmoid(priority_logits).numpy()
    department_preds = torch.softmax(department_logits, dim=1).numpy()
model.train()

##### The power of the Functional API: Access to layer connectivity

###### Plotting layer connectivity

In [ ]:
# PyTorch: there's no plot_model(); print(model) shows the module structure.
print(model)

In [ ]:
# PyTorch: no graph-plotting utility; torchinfo.summary(model, ...) would show
# per-layer output shapes. We just print the module here to keep deps minimal.
print(model)

###### Feature extraction with a Functional model

In [ ]:
# PyTorch: submodules are exposed via named_children()/named_modules().
list(model.named_children())

In [ ]:
# PyTorch: there are no stored symbolic input/output tensors per layer. The
# closest notion is the module that produces the features; inspect it directly.
model.dense_features

In [ ]:
# PyTorch: likewise, "output" of a layer is whatever its forward() returns at
# call time; there's no persistent symbolic output tensor to read here.
model.dense_features

In [ ]:
# PyTorch: "feature extraction" means reusing the shared layers and attaching a
# new head. We subclass to add a third output ("difficulty") on top of the same
# shared dense_features representation.
class TicketClassifierWithDifficulty(nn.Module):
    def __init__(self, base):
        super().__init__()
        self.dense_features = base.dense_features
        self.priority = base.priority
        self.department = base.department
        self.difficulty = nn.Linear(64, 3)

    def forward(self, title, text_body, tags):
        features = torch.cat([title, text_body, tags], dim=1)
        features = torch.relu(self.dense_features(features))
        priority = self.priority(features)
        department = self.department(features)
        difficulty = self.difficulty(features)
        return priority, department, difficulty

new_model = TicketClassifierWithDifficulty(model)

In [ ]:
# PyTorch: no plot_model(); inspect the extended module by printing it.
print(new_model)

#### Subclassing the Model class

##### Rewriting our previous example as a subclassed model

In [ ]:
# PyTorch: subclassing keras.Model maps directly to subclassing nn.Module.
# Concatenate -> torch.cat; the call() method becomes forward(). We return raw
# logits for both heads (no sigmoid/softmax) to pair with logit-based losses.
class CustomerTicketModel(nn.Module):
    def __init__(self, num_departments):
        super().__init__()
        self.mixing_layer = nn.Linear(
            vocabulary_size + vocabulary_size + num_tags, 64
        )
        self.priority_scorer = nn.Linear(64, 1)
        self.department_classifier = nn.Linear(64, num_departments)

    def forward(self, inputs):
        title = inputs["title"]
        text_body = inputs["text_body"]
        tags = inputs["tags"]

        features = torch.cat([title, text_body, tags], dim=1)
        features = torch.relu(self.mixing_layer(features))
        priority = self.priority_scorer(features)
        department = self.department_classifier(features)
        return priority, department

In [ ]:
model = CustomerTicketModel(num_departments=4)

priority, department = model(
    {"title": title_t, "text_body": text_body_t, "tags": tags_t}
)

In [ ]:
# PyTorch: same explicit train/eval/predict pattern as before, now feeding the
# inputs as a dict to match this model's forward() signature.
optimizer = torch.optim.Adam(model.parameters())
priority_loss_fn = nn.BCEWithLogitsLoss()
department_loss_fn = nn.CrossEntropyLoss()

batch_size = 32
for epoch in range(1):
    permutation = torch.randperm(num_samples)
    for i in range(0, num_samples, batch_size):
        idx = permutation[i : i + batch_size]
        inputs = {
            "title": title_t[idx],
            "text_body": text_body_t[idx],
            "tags": tags_t[idx],
        }
        priority_pred, department_pred = model(inputs)
        loss = priority_loss_fn(priority_pred, priority_t[idx]) + department_loss_fn(
            department_pred, department_t[idx]
        )
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

model.eval()
inputs = {"title": title_t, "text_body": text_body_t, "tags": tags_t}
with torch.no_grad():
    priority_pred, department_pred = model(inputs)
    mae = (torch.sigmoid(priority_pred) - priority_t).abs().mean().item()
    dept_acc = (department_pred.argmax(dim=1) == department_t).float().mean().item()
print(f"priority MAE: {mae:.4f}, department accuracy: {dept_acc:.4f}")

with torch.no_grad():
    priority_logits, department_logits = model(inputs)
    priority_preds = torch.sigmoid(priority_logits).numpy()
    department_preds = torch.softmax(department_logits, dim=1).numpy()
model.train()

##### Beware: What subclassed models don't support

#### Mixing and matching different components

In [ ]:
# PyTorch: both the subclassed component and the "functional" wrapper become
# nn.Modules. Classifier returns raw logits; the surrounding model composes it
# with a shared dense layer, just like nesting a layer inside keras.Model.
class Classifier(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        if num_classes == 2:
            num_units = 1
        else:
            num_units = num_classes
        self.dense = nn.LazyLinear(num_units)

    def forward(self, inputs):
        return self.dense(inputs)


class Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.dense = nn.Linear(3, 64)
        self.classifier = Classifier(num_classes=10)

    def forward(self, inputs):
        features = torch.relu(self.dense(inputs))
        return self.classifier(features)

model = Model()

In [ ]:
# PyTorch: a "functional" submodel is just another nn.Module we can hold as an
# attribute and call from forward(), exactly like nesting a keras.Model.
class BinaryClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.dense = nn.Linear(64, 1)

    def forward(self, inputs):
        return self.dense(inputs)


binary_classifier = BinaryClassifier()


class MyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.dense = nn.Linear(64, 64)
        self.classifier = binary_classifier

    def forward(self, inputs):
        features = torch.relu(self.dense(inputs))
        return self.classifier(features)

model = MyModel()

#### Remember: Use the right tool for the job

### Using built-in training and evaluation loops

In [ ]:
# PyTorch: the functional get_mnist_model() becomes a factory returning an
# nn.Module. Dropout -> nn.Dropout; we drop the softmax (CrossEntropyLoss wants
# logits). MNIST comes from torchvision instead of keras.datasets.
from torchvision.datasets import MNIST


def get_mnist_model():
    model = nn.Sequential(
        nn.Linear(28 * 28, 512),
        nn.ReLU(),
        nn.Dropout(0.5),
        nn.Linear(512, 10),
    )
    return model


mnist_train = MNIST(root="./data", train=True, download=True)
mnist_test = MNIST(root="./data", train=False, download=True)
images, labels = mnist_train.data.numpy(), mnist_train.targets.numpy()
test_images, test_labels = mnist_test.data.numpy(), mnist_test.targets.numpy()
images = images.reshape((60000, 28 * 28)).astype("float32") / 255
test_images = test_images.reshape((10000, 28 * 28)).astype("float32") / 255
train_images, val_images = images[10000:], images[:10000]
train_labels, val_labels = labels[10000:], labels[:10000]

# PyTorch: wrap the arrays in a DataLoader and write an explicit fit/evaluate
# loop with a per-epoch validation pass (validation_data was given).
from torch.utils.data import TensorDataset, DataLoader


def to_loader(x, y, batch_size=32, shuffle=False):
    ds = TensorDataset(torch.from_numpy(x), torch.from_numpy(y).long())
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)


def train_model(model, train_loader, val_loader, optimizer, loss_fn, epochs):
    for epoch in range(epochs):
        model.train()
        for xb, yb in train_loader:
            preds = model(xb)
            loss = loss_fn(preds, yb)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        val_loss, val_acc = evaluate_model(model, val_loader, loss_fn)
        print(f"Epoch {epoch}: val_loss {val_loss:.4f} val_acc {val_acc:.4f}")


def evaluate_model(model, loader, loss_fn):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for xb, yb in loader:
            preds = model(xb)
            total_loss += loss_fn(preds, yb).item() * len(yb)
            correct += (preds.argmax(dim=1) == yb).sum().item()
            total += len(yb)
    return total_loss / total, correct / total


train_loader = to_loader(train_images, train_labels, shuffle=True)
val_loader = to_loader(val_images, val_labels)
test_loader = to_loader(test_images, test_labels)

model = get_mnist_model()
optimizer = torch.optim.Adam(model.parameters())
loss_fn = nn.CrossEntropyLoss()
train_model(model, train_loader, val_loader, optimizer, loss_fn, epochs=3)
test_metrics = evaluate_model(model, test_loader, loss_fn)

# PyTorch: predict() -> no-grad forward producing probabilities.
model.eval()
with torch.no_grad():
    predictions = torch.softmax(
        model(torch.from_numpy(test_images)), dim=1
    ).numpy()

#### Writing your own metrics

In [ ]:
import torch.nn.functional as F

# PyTorch: a custom keras.metrics.Metric becomes a plain stateful class with
# update/result/reset methods. keras.ops.* -> torch.*. We accumulate the squared
# error sum and a sample count across batches, exactly like add_weight() state.
class RootMeanSquaredError:
    def __init__(self, name="rmse"):
        self.name = name
        self.reset_state()

    def update_state(self, y_true, y_pred, sample_weight=None):
        y_true = F.one_hot(y_true, num_classes=y_pred.shape[1]).float()
        mse = torch.sum(torch.square(y_true - y_pred))
        self.mse_sum += mse.item()
        self.total_samples += y_pred.shape[0]

    def result(self):
        return (self.mse_sum / self.total_samples) ** 0.5

    def reset_state(self):
        self.mse_sum = 0.0
        self.total_samples = 0

In [ ]:
# PyTorch: there's no metrics= argument; we update our custom metric inside the
# loop. The model outputs logits, so we softmax before feeding the RMSE metric
# (it expects probabilities, like the original).
model = get_mnist_model()
optimizer = torch.optim.Adam(model.parameters())
loss_fn = nn.CrossEntropyLoss()
rmse = RootMeanSquaredError()

for epoch in range(3):
    model.train()
    rmse.reset_state()
    for xb, yb in train_loader:
        preds = model(xb)
        loss = loss_fn(preds, yb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        rmse.update_state(yb, torch.softmax(preds, dim=1).detach())

    # PyTorch: validation pass with both accuracy and the custom RMSE metric.
    model.eval()
    val_rmse = RootMeanSquaredError()
    correct, total, val_loss = 0, 0, 0.0
    with torch.no_grad():
        for xb, yb in val_loader:
            preds = model(xb)
            val_loss += loss_fn(preds, yb).item() * len(yb)
            probs = torch.softmax(preds, dim=1)
            correct += (preds.argmax(dim=1) == yb).sum().item()
            total += len(yb)
            val_rmse.update_state(yb, probs)
    print(
        f"Epoch {epoch}: val_loss {val_loss / total:.4f} "
        f"val_acc {correct / total:.4f} val_rmse {val_rmse.result():.4f}"
    )

test_metrics = evaluate_model(model, test_loader, loss_fn)

#### Using callbacks

##### The EarlyStopping and ModelCheckpoint callbacks

In [ ]:
# PyTorch: callbacks have no built-in equivalent. We implement EarlyStopping
# (stop when the monitored metric stops improving for `patience` epochs) and
# ModelCheckpoint (torch.save the state_dict whenever val_loss improves) by hand
# inside the loop.
model = get_mnist_model()
optimizer = torch.optim.Adam(model.parameters())
loss_fn = nn.CrossEntropyLoss()

best_val_loss = float("inf")
best_train_acc = 0.0
patience, wait = 1, 0
checkpoint_path = "checkpoint_path.pt"

for epoch in range(10):
    model.train()
    correct, total = 0, 0
    for xb, yb in train_loader:
        preds = model(xb)
        loss = loss_fn(preds, yb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        correct += (preds.argmax(dim=1) == yb).sum().item()
        total += len(yb)
    train_acc = correct / total
    val_loss, val_acc = evaluate_model(model, val_loader, loss_fn)
    print(f"Epoch {epoch}: train_acc {train_acc:.4f} val_loss {val_loss:.4f}")

    # ModelCheckpoint(save_best_only=True, monitor="val_loss")
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), checkpoint_path)

    # EarlyStopping(monitor="accuracy", patience=1) on the training accuracy.
    if train_acc > best_train_acc:
        best_train_acc = train_acc
        wait = 0
    else:
        wait += 1
        if wait > patience:
            print("Early stopping")
            break

In [ ]:
# PyTorch: reload the best checkpoint into a fresh model via load_state_dict.
model = get_mnist_model()
model.load_state_dict(torch.load("checkpoint_path.pt"))

#### Writing your own callbacks

In [ ]:
from matplotlib import pyplot as plt

# PyTorch: a custom keras.callbacks.Callback becomes a small helper object we
# call manually at the matching points of our training loop (train begin, batch
# end, epoch end).
class LossHistory:
    def on_train_begin(self):
        self.per_batch_losses = []

    def on_batch_end(self, loss):
        self.per_batch_losses.append(loss)

    def on_epoch_end(self, epoch):
        plt.clf()
        plt.plot(
            range(len(self.per_batch_losses)),
            self.per_batch_losses,
            label="Training loss for each batch",
        )
        plt.xlabel(f"Batch (epoch {epoch})")
        plt.ylabel("Loss")
        plt.legend()
        plt.savefig(f"plot_at_epoch_{epoch}", dpi=300)
        self.per_batch_losses = []

In [ ]:
# PyTorch: drive the custom callback by calling its hooks at the right spots.
model = get_mnist_model()
optimizer = torch.optim.Adam(model.parameters())
loss_fn = nn.CrossEntropyLoss()

history = LossHistory()
history.on_train_begin()
for epoch in range(10):
    model.train()
    for xb, yb in train_loader:
        preds = model(xb)
        loss = loss_fn(preds, yb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        history.on_batch_end(loss.item())
    history.on_epoch_end(epoch)
    val_loss, val_acc = evaluate_model(model, val_loader, loss_fn)
    print(f"Epoch {epoch}: val_loss {val_loss:.4f} val_acc {val_acc:.4f}")

#### Monitoring and visualization with TensorBoard

In [ ]:
# PyTorch: keras.callbacks.TensorBoard -> torch.utils.tensorboard.SummaryWriter;
# we log scalars manually each epoch.
from torch.utils.tensorboard import SummaryWriter

model = get_mnist_model()
optimizer = torch.optim.Adam(model.parameters())
loss_fn = nn.CrossEntropyLoss()

writer = SummaryWriter(log_dir="/full_path_to_your_log_dir")
for epoch in range(10):
    model.train()
    running_loss, total = 0.0, 0
    for xb, yb in train_loader:
        preds = model(xb)
        loss = loss_fn(preds, yb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * len(yb)
        total += len(yb)
    val_loss, val_acc = evaluate_model(model, val_loader, loss_fn)
    writer.add_scalar("loss/train", running_loss / total, epoch)
    writer.add_scalar("loss/val", val_loss, epoch)
    writer.add_scalar("accuracy/val", val_acc, epoch)
writer.close()

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /full_path_to_your_log_dir

### Writing your own training and evaluation loops

#### Training vs. inference

#### Writing custom training step functions

##### A PyTorch training step function

In [ ]:
# PyTorch: a custom training step. Drop the %%backend magic and Keras wrappers;
# use a real nn.Module, nn.CrossEntropyLoss (logits) and torch.optim.Adam.
model = get_mnist_model()
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters())


def train_step(inputs, targets):
    model.train()
    predictions = model(inputs)
    loss = loss_fn(predictions, targets)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    return loss

In [ ]:
batch_size = 32
inputs = torch.from_numpy(train_images[:batch_size])
targets = torch.from_numpy(train_labels[:batch_size]).long()
loss = train_step(inputs, targets)

#### Low-level usage of metrics

In [ ]:
# PyTorch: there's no keras.metrics; sparse categorical accuracy is just the
# fraction of rows whose argmax matches the target. keras.ops.array -> torch.tensor.
targets = torch.tensor([0, 1, 2])
predictions = torch.tensor([[1, 0, 0], [0, 1, 0], [0, 0, 1]], dtype=torch.float32)
current_result = (predictions.argmax(dim=1) == targets).float().mean()
print(f"result: {current_result:.2f}")

In [ ]:
# PyTorch: keras.metrics.Mean -> a running average we maintain ourselves.
values = torch.tensor([0, 1, 2, 3, 4], dtype=torch.float32)
running_sum, count = 0.0, 0
for value in values:
    running_sum += value.item()
    count += 1
print(f"Mean of values: {running_sum / count:.2f}")

#### Using fit() with a custom training loop

##### Customizing fit() with PyTorch

In [ ]:
# PyTorch: "customizing fit()" means writing our own training-step method on an
# nn.Module and a loop that calls it. loss_tracker is a simple running mean.
loss_fn = nn.CrossEntropyLoss()


class RunningMean:
    def __init__(self, name="loss"):
        self.name = name
        self.reset()

    def reset(self):
        self.total, self.count = 0.0, 0

    def update_state(self, value):
        self.total += float(value)
        self.count += 1

    def result(self):
        return self.total / max(self.count, 1)


loss_tracker = RunningMean(name="loss")


class CustomModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = get_mnist_model()
        self.optimizer = torch.optim.Adam(self.parameters())

    def forward(self, inputs):
        return self.net(inputs)

    def train_step(self, data):
        inputs, targets = data
        predictions = self(inputs)
        loss = loss_fn(predictions, targets)
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()
        loss_tracker.update_state(loss)
        return {"loss": loss_tracker.result()}

    @property
    def metrics(self):
        return [loss_tracker]

In [ ]:
def get_custom_model():
    model = CustomModel()
    return model

In [ ]:
# PyTorch: the "fit" loop calls our custom train_step per batch.
model = get_custom_model()
for epoch in range(3):
    model.train()
    for metric in model.metrics:
        metric.reset()
    for xb, yb in train_loader:
        logs = model.train_step((xb, yb))
    print(f"Epoch {epoch}: loss {logs['loss']:.4f}")

#### Handling metrics in a custom train_step()

##### train_step() metrics handling with PyTorch

In [ ]:
# PyTorch: a custom train_step that also updates a list of metrics. compute_loss
# becomes our loss_fn; we iterate self.metrics to update and report each one.
class CustomModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = get_mnist_model()
        self.optimizer = torch.optim.Adam(self.parameters())
        self.loss_fn = nn.CrossEntropyLoss()
        self.loss_tracker = RunningMean(name="loss")
        self.acc_tracker = SparseCategoricalAccuracy(name="sparse_categorical_accuracy")

    def forward(self, inputs):
        return self.net(inputs)

    @property
    def metrics(self):
        return [self.loss_tracker, self.acc_tracker]

    def train_step(self, data):
        inputs, targets = data
        predictions = self(inputs)
        loss = self.loss_fn(predictions, targets)
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()

        for metric in self.metrics:
            if metric.name == "loss":
                metric.update_state(loss)
            else:
                metric.update_state(targets, predictions)

        return {m.name: m.result() for m in self.metrics}

In [ ]:
# PyTorch: helper accuracy metric matching keras.metrics.SparseCategoricalAccuracy.
class SparseCategoricalAccuracy:
    def __init__(self, name="sparse_categorical_accuracy"):
        self.name = name
        self.reset()

    def reset(self):
        self.correct, self.total = 0, 0

    def update_state(self, y_true, y_pred):
        self.correct += (y_pred.argmax(dim=1) == y_true).sum().item()
        self.total += len(y_true)

    def result(self):
        return self.correct / max(self.total, 1)


def get_custom_model():
    return CustomModel()


model = get_custom_model()
for epoch in range(3):
    model.train()
    for metric in model.metrics:
        metric.reset()
    for xb, yb in train_loader:
        logs = model.train_step((xb, yb))
    print(
        f"Epoch {epoch}: "
        + " ".join(f"{k} {v:.4f}" for k, v in logs.items())
    )